# S0 · Mapa de offsets de λ por spaxel (checklist G1)

**Spec:** [`docs/plan_wavesol_stripes_2026-07-17.md`](../docs/plan_wavesol_stripes_2026-07-17.md)  |  **Bloque:** S · wavesol/stripes  |  **Run de este set:** `ROXs42Bb_realigned`

Mide, spaxel a spaxel, el corrimiento espectral de las líneas de absorción de la primaria contra un espectro de referencia de campo (Xie+20 §4.2.2). Estructura alineada con slicers ⇒ diferencias de solución de λ por exposición/slice (*stripes*, Hashimoto+20). Es el **insumo formal de la decisión G1**.

| | |
|---|---|
| **Entrada** | `cube_telcorr.fits` (realineado) + ADP oficial de ESO (control independiente) |
| **Salida (QC/productos)** | `stages/stageS0_qc.json` (+ `stageS0_adp_qc.json`), `stages/stageS0_offset_map.fits`, `plots/s0_wavesol/` |
| **Consume aguas abajo** | **Decisión G1 (humana)** → Fase 2 (S2–S5) o cierre `fase2_descartable` |


## Qué hace S0 y cómo

Por cada spaxel del halo (selección por brillo, percentil 50) se normaliza el continuo por división de *running-median* (mata el continuo cromático del halo AO) en 4 ventanas de absorción estelar que **evitan** el láser AO, Hα (la primaria es emisora), y las bandas telúricas O₂/H₂O; se cross-correla contra el espectro de referencia del campo (`stripes._xcorr_shift_pixels`, subpíxel) y se toma la mediana de las ventanas usables. El resultado es un **mapa de offset** (Å) por spaxel.

**Corte S/N (lección del preliminar 2026-07-17):** cada spaxel lleva un error `err = σ_robusta(offsets_por_ventana)/√N`; el p95 del gate se calcula SOLO sobre spaxels con `err < max_err_ch` (0.08 ch ≈ 0.1 Å), porque el p95 crudo lo dominan spaxels débiles donde la xcorr falla (preliminar: p95 global 3.9 Å de puro ruido vs 0.18 Å en el núcleo r<1"). El perfil por columnas usa todos (su mediana ya es robusta).

La **métrica de estructura** colapsa el mapa a lo largo de la dirección de los stripes (perfil por columna) y compara su amplitud contra el ruido esperado; el perfil **transversal** es el control: stripes reales muestran estructura en el perfil de stripe, no en el transversal.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.qc.wavesol_map --run-id $RUN --orientation vertical
```

Coste: full-res 330×338, normalización vectorizada; ~minutos por cubo.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stageS0_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.qc.wavesol_map --run-id $RUN --orientation vertical'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stageS0_qc.json', RUN_ID)
nb.show(qc, keys=['gate_g1.recommendation', 'gate_g1.decision', 'metrics.p95_abs_offset_A', 'metrics.structure_significance', 'metrics.transverse_significance', 'metrics.n_selected_low_err', 'channel_step_A', 'runtime_s'], title='S0')


## Evidencia: realineado vs ADP (control)

Los dos cubos deben coincidir: el mapa de offset es un diagnóstico del **instrumento/reducción**, no del cubo concreto. El preliminar 2×2 (2026-07-17) dio amplitud de columna 72 mÅ (realineado) / 67 mÅ (ADP), a ~1× ruido, sin estructura alineada con slicers; el núcleo r<1" med|off| = 64 / 62 mÅ ≈ el M1 global (+74 mÅ).

**Full-res reproduce la conclusión clave** (sin estructura de slicer: `stripe_sig` ≈ control transversal) y la extiende: al medir TODOS los spaxels de bajo error (no solo el núcleo) el p95 sube a ~0.32 Å — un scatter de λ por spaxel, consistente entre ventanas pero **espacialmente desestructurado**. La celda imprime ambos cubos; deben coincidir.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('S0', 'stages/stageS0_qc.json'):
        # p95 se mide sobre el subconjunto de bajo error (err<max_err_ch); la
        # estructura de slicer se juzga con stripe_sig vs el control transversal.
        rows = []
        for label, rel in [('realineado', 'stages/stageS0_qc.json'),
                           ('ADP',        'stages/stageS0_adp_qc.json')]:
            try:
                q = nb.load_qc(rel, RUN_ID)
            except FileNotFoundError:
                print(f'[{label}] QC aún no existe: {rel}'); continue
            m = q['metrics']
            rows.append((label, m['p95_abs_offset_A'], m['structure_significance'],
                         m['transverse_significance'], m['n_selected_low_err'],
                         m['n_spaxels_measured'], q['gate_g1']['recommendation']))
        hdr = ('cubo', 'p95|off|[A]', 'stripe_sig', 'transv_sig', 'n_low_err',
               'n_meas', 'recomendación')
        print('{:>10} {:>12} {:>11} {:>11} {:>10} {:>8}  {}'.format(*hdr))
        for r in rows:
            print('{:>10} {:>12.4f} {:>11.2f} {:>11.2f} {:>10d} {:>8d}  {}'.format(*r))
        print('\np95 sobre spaxels de bajo error; stripe_sig<=transv_sig => sin estructura de slicer.')


## Checklist G1 — umbrales y argumentos

**Umbrales del plan (recomendación automática, decisión humana):**

| Criterio | Umbral | Dispara Fase 2 si |
|---|---|---|
| p95 \|offset\| (spaxels `err<0.1 Å`) | 0.1 Å | **>** 0.1 Å |
| Estructura alineada con slicers | 3× ruido | **>** 3× **y** > 2× el control transversal |

**Argumentos del caso `fase2_descartable`** — registro de la decisión tomada para **ROXs 12 b** el 2026-07-17, con las cifras que se tenían entonces; se reproduce literal por trazabilidad y **no** describe a otro objeto (para el tuyo, los valores vivos están en su A4 y en el QC de S0):

1. La métrica espacial de S0 **no ve un offset común a todas las exposiciones** (deriva temporal uniforme): ese modo no aparece como estructura espacial, solo **ensancharía la LSF combinada**.
2. Pero A4·M2 midió **LSF = 2.383 Å**, MÁS ESTRECHA que el nominal → acota ese *smearing* a nivel pequeño (si hubiera deriva grande entre exposiciones, la LSF combinada saldría ensanchada, no estrecha).
3. El **M1 global (+0.074 Å)** ya corrige el zero-point de λ.

Los tres juntos son el caso para **no** entrar a la Fase 2 (re-reducción por exposición). La decisión final es **humana** (gate G1).


## Mapas S0 — panorama de TODOS los cubos (7 exposiciones + combinado + ADP)

Renderizados **directamente del FITS** `stageS0_offset_map.fits`. Se incluyen las **7 exposiciones individuales** (S2, `/mnt/2TB/MUSE_work/ROXs12b_perexp/`), el **combinado** (realineado) y el **ADP** de ESO (control) — 9 casos. Primero el detalle de 4 paneles del combinado (offset/error/perfil-columna/histograma), luego una rejilla 3×3 con el mapa de offset de cada caso.

**Por qué mirar por exposición:** las cabeceras (`INS DROT MODE=SKY`, `INS DROT POSANG` = 0°,0°,90°,90°,180°,180°,0°) muestran un **patrón deliberado de rotación de 90°** entre grupos de exposiciones; los 7 cubos están remuestreados norte-arriba ⇒ el combinado mezcla TRES asignaciones de slice distintas (0°/90°/180°) y el scrambling de un stripe fijo del slicer es total. Cada exposición sí conserva su orientación de slicer (vertical para POSANG 0/180, **horizontal para 90** — exp3/exp4), así que 0/7 con estructura, medido cada uno en su eje, es la prueba real de que no hay stripes.


In [ ]:
try:
    import os, numpy as np, matplotlib.pyplot as plt
    from astropy.io import fits
    import musepipe.qc.wavesol_map as wsm
    rd = nb.run_dir(RUN_ID)
    # Directorio de cubos por exposición: del config del run, no fijo (WP-E4b).
    import sys as _sys; _sys.path.insert(0, str(nb.project_root()))
    from musepipe.config import run_workdir_setting
    PEREXP_DIR = run_workdir_setting(RUN_ID, 'perexp_dir', project_root=nb.project_root())
    # CASES: (label, qc_path, map_path)  — per-exp absolutos; combinado/ADP en el run
    CASES = [(f'exp{i}', f'{PEREXP_DIR}/exp{i}/stageS0_qc.json',
              f'{PEREXP_DIR}/exp{i}/stageS0_offset_map.fits') for i in range(1, 8)]
    CASES += [('combinado', str(rd/'stages'/'stageS0_qc.json'), str(rd/'stages'/'stageS0_offset_map.fits')),
              ('ADP',       str(rd/'stages'/'stageS0_adp_qc.json'), str(rd/'stages'/'stageS0_adp_offset_map.fits'))]
    import json
    def _load(qcf, mapf):
        q = json.load(open(qcf)); step = q['channel_step_A']; thr = q['gate_g1']['thresholds']['max_err_ch']
        with fits.open(mapf) as h:
            d = {k: h[k].data.astype(float) for k in ('OFFSET_A','ERR_A','OFFSET_CH','ERR_CH')}
        return q, step, thr, d
    # --- detalle 4-panel del combinado ---
    cm = [c for c in CASES if c[0]=='combinado'][0]
    q, step, thr, d = _load(cm[1], cm[2])
    low = np.isfinite(d['OFFSET_A']) & np.isfinite(d['ERR_CH']) & (d['ERR_CH'] < thr)
    prof = wsm.stripe_profile(d['OFFSET_CH'], 'vertical')
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6)); fig.suptitle(f'S0 · combinado (detalle) — {int(low.sum())} spaxels bajo error', fontsize=11)
    vl = np.nanpercentile(np.abs(d['OFFSET_A'][low]), 95)
    im0 = ax[0].imshow(d['OFFSET_A'], origin='lower', cmap='RdBu_r', vmin=-vl, vmax=vl); ax[0].set_title('offset (Å)'); plt.colorbar(im0, ax=ax[0], fraction=0.046)
    im1 = ax[1].imshow(d['ERR_A'], origin='lower', cmap='viridis', vmax=np.nanpercentile(d['ERR_A'],95)); ax[1].set_title('error (Å)'); plt.colorbar(im1, ax=ax[1], fraction=0.046)
    ax[2].plot(np.arange(prof['profile'].size), np.asarray(prof['profile'],float)*step, lw=0.9); ax[2].axhline(0,color='0.6',lw=0.6); ax[2].set_title('perfil por columna (∥ stripes)'); ax[2].set_xlabel('columna'); ax[2].set_ylabel('offset mediano (Å)')
    ax[3].hist(d['OFFSET_A'][low], bins=60, color='tab:blue', alpha=0.8); ax[3].axvline(0,color='k',lw=0.8); ax[3].axvline(np.median(d['OFFSET_A'][low]),color='tab:red',ls='--',label=f"mediana {np.median(d['OFFSET_A'][low])*1e3:+.0f} mÅ"); ax[3].set_title('hist (bajo error)'); ax[3].set_xlabel('offset (Å)'); ax[3].legend(fontsize=8)
    fig.tight_layout(); plt.show()
    # --- rejilla 3x3 de mapas de offset (9 casos) ---
    fig, axes = plt.subplots(3, 3, figsize=(13, 12)); fig.suptitle('S0 · mapa de offset (Å) por caso — 7 exposiciones + combinado + ADP', fontsize=12)
    for axi, (label, qcf, mapf) in zip(axes.ravel(), CASES):
        if not os.path.exists(mapf):
            axi.set_title(f'{label}: (falta)'); axi.axis('off'); continue
        _, _, thr, d = _load(qcf, mapf)
        lo = np.isfinite(d['OFFSET_A']) & np.isfinite(d['ERR_CH']) & (d['ERR_CH'] < thr)
        vl = np.nanpercentile(np.abs(d['OFFSET_A'][lo]), 95) if lo.any() else 0.3
        im = axi.imshow(d['OFFSET_A'], origin='lower', cmap='RdBu_r', vmin=-vl, vmax=vl)
        axi.set_title(f'{label}  (n={int(lo.sum())}, ±{vl*1e3:.0f} mÅ)', fontsize=9)
        plt.colorbar(im, ax=axi, fraction=0.046)
    fig.tight_layout(); plt.show()
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## Paso extra — crop 100×100 en la estrella, para los 9 casos

**Motivación:** el gate global lo lastran los spaxels débiles del borde (el p95 crudo y el `median|offset|` los dominan). Si hubiera un offset **coherente escondido en el ruido**, se saca a la luz midiéndolo donde la S/N es máxima: el halo de la estrella. Se recorta un **100×100 centrado en la estrella** (centroide de menor error, sin cargar el cubo) y se recalculan las métricas + el **offset medio con signo ± error** — el test directo de un offset coherente oculto (que `median|·|` no ve por no distinguir signo).

Se hace para **los 7 cubos por exposición + combinado + ADP** (full vs crop), midiendo el stripe **en la orientación real del slicer de cada caso** según `INS DROT POSANG` (vertical para 0/180, horizontal para 90 — exp3/exp4; corrección 2026-07-19, ver `decision_g1_wavesol_2026-07-17.md`). Dos lecturas: (1) el **offset medio del crop por exposición** traza la **deriva temporal** de zero-point (modo TEMPORAL de G1/S3); (2) `stripe_sig` ≤ el control perpendicular en todos ⇒ **ningún stripe de slicer** aun al máximo S/N (modo ESPACIAL descartado).

Salida: la tabla, la figura resumen (deriva + stripe/transversal) y, para cada caso, el **mismo detalle de 4 paneles que la celda de mapas pero recortado al crop** (offset, error, perfil por columna, histograma).


In [ ]:
try:
    import os, json, numpy as np, matplotlib.pyplot as plt
    from astropy.io import fits
    import musepipe.qc.wavesol_map as wsm
    rd = nb.run_dir(RUN_ID); HALF = 50
    import sys as _sys; _sys.path.insert(0, str(nb.project_root()))
    from musepipe.config import run_workdir_setting
    PEREXP_DIR = run_workdir_setting(RUN_ID, 'perexp_dir', project_root=nb.project_root())
    CASES = [(f'exp{i}', f'{PEREXP_DIR}/exp{i}/stageS0_qc.json',
              f'{PEREXP_DIR}/exp{i}/stageS0_offset_map.fits') for i in range(1, 8)]
    CASES += [('combinado', str(rd/'stages'/'stageS0_qc.json'), str(rd/'stages'/'stageS0_offset_map.fits')),
              ('ADP',       str(rd/'stages'/'stageS0_adp_qc.json'), str(rd/'stages'/'stageS0_adp_offset_map.fits'))]
    # Orientación del slicer EN el cubo norte-arriba, por INS DROT POSANG (modo SKY):
    # POSANG 0/180 -> vertical; POSANG 90 -> horizontal. Combinado/ADP: vertical (dominante 5/7).
    POSANG = {'exp1': 0, 'exp2': 0, 'exp3': 90, 'exp4': 90, 'exp5': 180, 'exp6': 180, 'exp7': 0}
    orient_of = lambda label: 'horizontal' if POSANG.get(label, 0) == 90 else 'vertical'
    def stats(oc, ec, oa, step, thr, orient):
        m = wsm.structure_metrics(oc, step, orientation=orient, err_map_ch=ec, max_err_ch=thr)
        low = np.isfinite(oa) & np.isfinite(ec) & (ec < thr); v = oa[low]
        mean = float(np.mean(v)); se = float(np.std(v) / np.sqrt(max(v.size, 1)))
        return m, mean, se
    print('stripe/ctrl medidos en la orientación del slicer de CADA caso (POSANG):')
    print('{:>10} {:>5} {:>6} {:>8} {:>7} {:>9} {:>6} {:>6}  {}'.format(
          'caso/región','n_lowE','med|o|','p95 Å','meanmÅ','σ_mean','strp','ctrl','orient'))
    rows = []; crops = []
    for label, qcf, mapf in CASES:
        if not os.path.exists(mapf):
            print(f'{label:>10}  (falta {mapf})'); continue
        q = json.load(open(qcf)); step = q['channel_step_A']; thr = q['gate_g1']['thresholds']['max_err_ch']
        with fits.open(mapf) as h:
            oc = h['OFFSET_CH'].data.astype(float); oa = h['OFFSET_A'].data.astype(float); ec = h['ERR_CH'].data.astype(float)
        ny, nx = oc.shape; low = np.isfinite(oc) & np.isfinite(ec) & (ec < thr)
        if low.sum() < 10:
            print(f'{label:>10}  (pocos spaxels de bajo error)'); continue
        yy, xx = np.mgrid[0:ny, 0:nx]; wt = np.where(low, 1.0/np.clip(ec, 1e-3, None)**2, 0.0)
        cy = int(round(np.sum(yy*wt)/np.sum(wt))); cx = int(round(np.sum(xx*wt)/np.sum(wt)))
        sl = (slice(max(0,cy-HALF),min(ny,cy+HALF)), slice(max(0,cx-HALF),min(nx,cx+HALF)))
        orient = orient_of(label)
        mF, meanF, seF = stats(oc, ec, oa, step, thr, orient)
        mC, meanC, seC = stats(oc[sl], ec[sl], oa[sl], step, thr, orient)
        for tag, mm, mn, se in [('FULL', mF, meanF, seF), ('CROP', mC, meanC, seC)]:
            print('{:>10} {:>5d} {:>6.0f} {:>8.4f} {:>+7.1f} {:>9.1f} {:>6.2f} {:>6.2f}  {}'.format(
                  f'{label} {tag}', mm['n_selected_low_err'], mm['median_abs_offset_ch']*step*1e3,
                  mm['p95_abs_offset_A'], mn*1e3, se*1e3, mm['structure_significance'], mm['transverse_significance'], orient[:4]))
        rows.append((label, meanC*1e3, seC*1e3, mC['structure_significance'], mC['transverse_significance']))
        crops.append((label, oc[sl].copy(), oa[sl].copy(), ec[sl].copy(), step, thr, meanC, seC, orient))
    # --- figura resumen: deriva temporal + stripe vs transversal ---
    exps = [r for r in rows if r[0].startswith('exp')]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    if exps:
        x = np.arange(len(exps)); labs = [r[0] for r in exps]
        ax[0].errorbar(x, [r[1] for r in exps], yerr=[r[2] for r in exps], fmt='o', color='tab:blue', capsize=3, label='crop mean por exp')
        for name, col in [('combinado','tab:green'), ('ADP','tab:red')]:
            rr = [r for r in rows if r[0]==name]
            if rr: ax[0].axhline(rr[0][1], color=col, ls='--', lw=1, label=f'{name} {rr[0][1]:+.0f} mÅ')
        ax[0].axhline(0, color='0.6', lw=0.6); ax[0].axhline(74, color='0.4', ls=':', lw=1, label='M1 global +74 mÅ')
        ax[0].set_xticks(x); ax[0].set_xticklabels(labs); ax[0].set_ylabel('offset medio del crop (mÅ)')
        ax[0].set_title('Deriva temporal del zero-point (crop alta S/N)'); ax[0].legend(fontsize=7)
    for r in rows:
        mk = 'o' if r[0].startswith('exp') else ('s' if r[0]=='combinado' else '^')
        ax[1].scatter(r[4], r[3], marker=mk, s=55); ax[1].annotate(r[0], (r[4], r[3]), fontsize=7, xytext=(3,3), textcoords='offset points')
    limmax = 3.0
    ax[1].plot([0,limmax],[0,limmax], color='0.6', ls='--', lw=1, label='stripe = transversal')
    ax[1].axhline(3, color='tab:red', ls=':', lw=1, label='umbral stripe 3×')
    ax[1].set_xlabel('control (⊥ slicer)'); ax[1].set_ylabel('stripe_sig (∥ slicer, orientación por POSANG)'); ax[1].set_xlim(0,limmax); ax[1].set_ylim(0,limmax)
    ax[1].set_title('stripe vs control (bajo la diagonal = sin stripe)'); ax[1].legend(fontsize=7)
    fig.tight_layout(); plt.show()
    # --- detalle 4-panel del CROP por caso (como la celda de mapas, pero recortado) ---
    print('\nDetalle 4-panel del crop (offset / error / perfil ∥ slicer / histograma) por caso:')
    for label, oc_c, oa_c, ec_c, step, thr, meanC, seC, orient in crops:
        lo = np.isfinite(oa_c) & np.isfinite(ec_c) & (ec_c < thr)
        prof = wsm.stripe_profile(oc_c, orient)
        fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
        fig.suptitle(f'S0 · {label} · CROP 100×100 (slicer {orient}; n bajo error={int(lo.sum())}, media {meanC*1e3:+.1f}±{seC*1e3:.1f} mÅ)', fontsize=10)
        vl = np.nanpercentile(np.abs(oa_c[lo]), 95) if lo.any() else 0.3
        im0 = ax[0].imshow(oa_c, origin='lower', cmap='RdBu_r', vmin=-vl, vmax=vl); ax[0].set_title('offset (Å)'); plt.colorbar(im0, ax=ax[0], fraction=0.046)
        im1 = ax[1].imshow(ec_c*step, origin='lower', cmap='viridis', vmax=np.nanpercentile(ec_c[np.isfinite(ec_c)]*step, 95) if np.isfinite(ec_c).any() else None); ax[1].set_title('error (Å)'); plt.colorbar(im1, ax=ax[1], fraction=0.046)
        p_A = np.asarray(prof['profile'], float) * step
        ax[2].plot(np.arange(p_A.size), p_A, lw=0.9); ax[2].axhline(0, color='0.6', lw=0.6); ax[2].set_title(f'perfil ∥ slicer ({orient})'); ax[2].set_xlabel('posición transversal (crop)'); ax[2].set_ylabel('offset mediano (Å)')
        if lo.any():
            ax[3].hist(oa_c[lo], bins=40, color='tab:green', alpha=0.8); ax[3].axvline(0, color='k', lw=0.8)
            ax[3].axvline(meanC, color='tab:red', ls='--', lw=1.2, label=f'media {meanC*1e3:+.1f} mÅ ({abs(meanC/seC):.0f}σ)'); ax[3].legend(fontsize=8)
        ax[3].set_title('hist (bajo error)'); ax[3].set_xlabel('offset (Å)')
        fig.tight_layout(); plt.show()
    print('\nLectura: (1) el offset medio del crop deriva exposición a exposición (~±40 mÅ, el modo')
    print('TEMPORAL de S3, ≲0.04 canal); (2) todos los casos caen en/bajo la diagonal stripe=transversal')
    print('y muy por debajo del umbral 3× => NINGÚN stripe de slicer, ni al máximo S/N. Refuerza el cierre G1.')
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## Decisiones y notas
- S0a/S0b: núcleo target-agnostic (`musepipe/qc/wavesol_map.py`) + CLI, normalización vectorizada (gate de equivalencia <1e-9) y corte S/N por spaxel; 14 tests verdes. · [`docs/plan_wavesol_stripes_pasos_agente.md`](../docs/plan_wavesol_stripes_pasos_agente.md)
- El offset map es diagnóstico del instrumento/reducción: realineado ≈ ADP (control cruzado). · [`docs/plan_wavesol_stripes_2026-07-17.md`](../docs/plan_wavesol_stripes_2026-07-17.md)
- **Gate G1 (humano):** con este producto se decide entrar o no a la Fase 2 (S2–S5, re-reducción por exposición). La recomendación automática se imprime en Checks. · [`docs/plan_wavesol_stripes_2026-07-17.md`](../docs/plan_wavesol_stripes_2026-07-17.md)
- **DECIDIDO 2026-07-17:** cerrar como sistemático acotado (interpretación temporal); Fase 2 NO disparada; confirmación diferida (S0 por exposición cuando existan los 7 cubos). Anula la recomendación automática. · [`docs/decision_g1_wavesol_2026-07-17.md`](../docs/decision_g1_wavesol_2026-07-17.md)
- **Corrección POSANG 2026-07-19 (hallazgo del usuario):** `INS DROT POSANG` = 0/0/90/90/180/180/0 en modo SKY (el ABSROT −16→+6° es el ángulo físico del derotador, no rotación de campo). exp3/exp4 tienen el slicer HORIZONTAL en el cubo norte-arriba: sus métricas stripe/control estaban intercambiadas. Con la orientación correcta el veredicto NO cambia (0/7 sin stripes); la ceguera del combinado se refuerza (scrambling total, no smear de 5.9°). Tabla corregida: `tables/s0_perexp_summary_posang.csv`. · [`docs/decision_g1_wavesol_2026-07-17.md`](../docs/decision_g1_wavesol_2026-07-17.md)


## Checks


In [ ]:
try:
    q = nb.load_qc('stages/stageS0_qc.json', RUN_ID)
    m, g = q['metrics'], q['gate_g1']
    th = g['thresholds']
    p95 = m['p95_abs_offset_A']; sig = m['structure_significance']; sigt = m['transverse_significance']
    c1 = p95 <= th['p95_threshold_A']
    aligned = (sig > th['significance_threshold']) and (sigt != sigt or sig > 2.0 * sigt)
    print('CHECKLIST G1 (realineado, full-res):')
    print(f"  p95|off| = {p95:.4f} A  (umbral {th['p95_threshold_A']} A)   -> {'OK pequeño' if c1 else 'GRANDE'}")
    print(f"  estructura stripe = {sig:.2f}x ruido   (control transversal {sigt:.2f}x, umbral {th['significance_threshold']}x)")
    print(f"  {'sin' if not aligned else 'CON'} estructura alineada con slicers dominante")
    print(f"  n_spaxels bajo corte err<{th['max_err_ch']} ch = {m['n_selected_low_err']} / {m['n_spaxels_measured']} medidos")
    print(f"\n  recomendación automática: {g['recommendation']}")
    for r in g['reasons']:
        print('   -', r)
    hd = q.get('g1_human_decision')
    if hd:
        print(f"\n  DECISIÓN G1 (humana, {hd['date']}): {hd['decision']} "
              f"[interpretación: {hd['interpretation']}; anula {hd['recommendation_overridden']}]")
        print(f"    fase2_triggered = {hd['phase2_triggered']}; confirmación diferida: {hd['deferred_confirmation'][:80]}...")
        print(f"    doc: {hd['doc']}")
    else:
        print(f"  decisión: {g['decision']} (aún no registrada en este QC)")
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## Conclusión (registrada, 2026-07-17) — G1 DECIDIDO

> Registro de la decisión tomada para **ROXs 12 b** (cifras de esa fecha, literales por trazabilidad). Si estás en el set de otro objeto, esta conclusión **no** es la suya: la de tu objeto sale de su propio `stageS0_qc.json`, arriba.

**S0 (full-res, 330×338):** sin estructura de slicer (`stripe_sig` ≤ control transversal) y **realineado ≈ ADP**, pero p95\|off\| ≈ 0.32 Å (>0.1 Å), scatter por spaxel creciente con el radio.

**Clave (por qué el cubo combinado es ciego a los stripes) — corregido 2026-07-19 (POSANG):** las 7 exposiciones son de una noche, dithers ≈0, pero `INS DROT POSANG` = **0°,0°,90°,90°,180°,180°,0°** (modo SKY: PA fijo en cielo durante cada exposición; el `ABSROT` −16→+6° es el ángulo físico del derotador, no rotación de campo). El combinado mezcla **tres asignaciones de slice** (0°/90°/180°) ⇒ el scrambling de un stripe fijo del slicer es **total** (cuerda de 90° en el compañero ≈ 2.5″), y aparece como scatter desestructurado. **S0 sobre el combinado no puede confirmar ni descartar stripes**; el 'sin estructura' es esperable en cualquier caso. En los cubos por exposición el slicer queda **vertical para POSANG 0/180 y horizontal para 90 (exp3/exp4)** — la celda del crop mide cada caso en su orientación.

➡️ **DECISIÓN G1 (humana, 2026-07-17): cerrar como sistemático acotado, interpretación TEMPORAL** (deriva de zero-point por exposición, acotada por A4·M2 LSF=2.383 Å). **Fase 2 NO disparada.** Anula la recomendación automática (`fase2_justificada`, que salía solo por el p95). **Salvedad:** M2 acota el modo temporal uniforme, no los stripes rotados; defendible para la no-detección de Hα / límites (E1/E3), más débil para líneas finas en G2/G3.

**Confirmación (HECHA 2026-07-18):** se regeneraron los 7 cubos por exposición (S2) y se corrió **S0 por exposición** (0/7 con estructura de slicer) **+ S3** (deriva temporal ~0.04 Å std). Ambas ramas cerradas ⇒ **GATE G1 CERRADO** (`gate_g1.decision=closed`). Detalle en `docs/decision_g1_wavesol_2026-07-17.md`.

**Corroboración con el crop 100×100 en la estrella (paso extra):** al restringir a la región de máxima S/N, el `median|offset|` cae de ~0.8 Å a ~0.11 Å (era ruido de bordes) y el offset medio con signo se resuelve a **unas decenas de mÅ** (realineado ≈ −48 mÅ, ADP ≈ −11 mÅ; ≲0.04 canal), pero **`stripe_sig` sigue ≤ el control transversal** en ambos cubos → **ningún stripe de slicer emerge al subir la S/N**. Lo que queda es un zero-point casi uniforme, del orden de la deriva temporal S3 (~0.04 Å) y dentro de lo que M1 (+74 mÅ) corrige — refuerza el cierre G1 al mejor S/N disponible.
